# Handling Sequences with Pytorch

- Sequential data: ordered in time or space, examples:

    * Time series
    * Text
    * Audio waves

- Traing - test split method: split by time.

- Predict single next data point based on specific length of previous data points.

- The task for this chapter is to predict future electricity consumption based on past patterns.

## 0 - Import Libraries

In [10]:
import numpy as np
import polars as pl
import polars.selectors as sc
import torch
from torch.utils.data import TensorDataset

## 1 - Load Data

[ElectricityLoadDiagrams20112014](https://archive.ics.uci.edu/dataset/321/electricityloaddiagrams20112014). This data set contains electricity consumption of 370 points/clients, view the data folder.

Values are in kW of each 15 min. To convert values in kWh values must be divided by 4. Each column represent one client. View the UC Irvine Machine Learing Repository for more details.


In [2]:
filepath = r'/home/devenv/Projects/deep-learning-with-pytorch/01-intermediate-dl/data/electricity_data/LD2011_2014.txt'

df_electricity = (
    pl.read_csv(
        source=filepath,
        columns=['', 'MT_001'],
        separator=";",
        decimal_comma=True,
        use_pyarrow=True
    )
    .rename(mapping={'column_0': 'timestamp'})
    .with_columns(
        sc.string().str.replace(',', '.')
        .cast(pl.Float32)
    )
)

In [3]:
# Exploratory data analysis
# df.glimpse()
df_electricity.tail()

timestamp,MT_001
datetime[ms],f32
2014-12-31 23:00:00,2.538071
2014-12-31 23:15:00,2.538071
2014-12-31 23:30:00,2.538071
2014-12-31 23:45:00,1.269036
2015-01-01 00:00:00,2.538071


## 2 - Traint, test split

* To be able to train neural networks on sequential data, you need to pre-process it first. You'll chunk the data into inputs-target pairs, where the inputs are some number of consecutive data points and the target is the next data point (DataCamp Team).

* Avoid look-ahead bias, emphasizing the need to split sequential data by time for training and testing.

In [4]:
def create_sequences(df: pl.DataFrame, seq_length: int):
    xs, ys = [], []
    
    for i in range(len(df) - seq_length):

        # Define inputs
        x = df[i: i + seq_length, 1]
        
        # Define outputs
        y = df[i + seq_length, 1]
        
        xs.append(x)
        ys.append(y)
        
    return np.array(xs), np.array(ys)

Create training examples:

In [5]:
training_samples = 24 * 4 # Considering last 24 hours
X_train, y_train = create_sequences(df=df_electricity, seq_length=training_samples)
print(f'Predictors: {X_train.shape}\nTarget: {y_train.shape}')

Predictors: (140160, 96)
Target: (140160,)


In [8]:
# Example
print(X_train[50000]) # Sequence length
y_train[50000] # Singel target

[17.766497  16.497461  16.497461  17.766497  16.497461  17.766497
 17.766497  17.766497  17.766497  17.766497  16.497461  13.959391
  7.614213  17.766497   6.3451777  5.0761423  3.8071065  5.0761423
  5.0761423  3.8071065  5.0761423  3.8071065  5.0761423  3.8071065
  5.0761423  5.0761423  3.8071065  5.0761423  3.8071065  5.0761423
  5.0761423  3.8071065  5.0761423  5.0761423  5.0761423  3.8071065
  5.0761423  5.0761423  3.8071065  5.0761423  3.8071065  3.8071065
  5.0761423  3.8071065  3.8071065  3.8071065  3.8071065 15.228426
 15.228426  16.497461  16.497461  15.228426  15.228426  16.497461
 15.228426  15.228426  15.228426  15.228426  16.497461  20.30457
 25.38071   15.228426  15.228426  13.959391  15.228426  13.959391
 15.228426  16.497461  13.959391  15.228426  13.959391  15.228426
 13.959391  13.959391  13.959391  15.228426  16.497461  13.959391
 12.690355   7.614213   6.3451777  2.5380712  1.2690356  2.5380712
  2.5380712  1.2690356  2.5380712  2.5380712  1.2690356  2.5380712
  2.

np.float64(2.5380711555480957)

Convert to Torch Dataset:

In [12]:
dataset_train = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).float()
)

print(len(dataset_train))

140160


## 3 - Recurrent Neural Networks

Review Chapter 15 from Machine Learning with PyTorch and Scikit Learn book.